# 07 Two Crystal Bowls

**Series:** Physics of the Integratron (Landers, California)
**Topic T7:** what you hear when two crystal bowls sing at once, and how the dome changes it.

Sound baths often use quartz crystal bowls, played by rubbing a mallet round the rim so that the bowl sings with a steady tone. When two bowls play notes that are close but not identical, the sound swells and fades in a slow, regular pulse. This notebook builds that sound from its parts.

1. **One crystal bowl.** A steady tone with a few faint overtones.
2. **Two bowls: beats.** Two close notes add up to one note that throbs.
3. **Two bowls around a listener.** In stereo the throb moves between the ears. Also what "binaural beats" are.
4. **Two bowls in the dome.** The dome's resonances can deepen the throb or erase it.

The bowl sounds are synthesized from a simple model. The pitches, overtone strengths and envelopes are assumed values, chosen to be typical, not measured from particular bowls. Use headphones for section 3.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from ipywidgets import interact, FloatSlider
from scipy.signal import lfilter

SPEED_OF_SOUND = 343.0                   # m/s, air at 20 °C
DOME_RADIUS = 55 * 0.3048 / 2            # 55 ft diameter -> 8.38 m
R = DOME_RADIUS
ROUND_TRIP_S = 2 * R / SPEED_OF_SOUND
AUDIO_RATE = 22050

BOWL_1_HZ = 261.6                        # assumed: a bowl tuned to C4
BOWL_2_HZ = 263.0                        # assumed: a second C bowl, 1.4 Hz sharp
# Overtones of a rubbed crystal bowl: ring ratios from notebook 05, assumed to be faint
OVERTONE_RATIOS = [1.0, 2.83, 5.42]
OVERTONE_LEVELS = [1.0, 0.12, 0.04]

BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8a85"
SLIDER_STYLE = {"handle_color": BLUE}  # visible knob on white backgrounds
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "lines.linewidth": 2,
})


def bowl_envelope(t, swell_s=2.0, stop_s=8.0, release_t60_s=12.0):
    # Rubbing builds the tone up, holds it, then the bowl rings down after the mallet stops
    rise = np.minimum(1, t / swell_s) ** 2
    fall = np.where(t < stop_s, 1.0, 10 ** (-3 * (t - stop_s) / release_t60_s))
    return rise * fall


def crystal_bowl(fundamental_hz, t, delay_s=0.0):
    # One bowl, heard delay_s after it sounds
    sound = np.zeros_like(t)
    for ratio, level in zip(OVERTONE_RATIOS, OVERTONE_LEVELS):
        sound += level * np.cos(2 * np.pi * fundamental_hz * ratio * (t - delay_s))
    return sound * bowl_envelope(np.maximum(t - delay_s, 0))


def play(samples):
    # Mono, or stereo as two rows, scaled so the loudest moment is 0.9
    display(Audio(samples / np.abs(samples).max() * 0.9, rate=AUDIO_RATE, normalize=False))


T = np.arange(0, 12.0, 1 / AUDIO_RATE)

## 1. One crystal bowl

A rubbed bowl sings mostly at its lowest rim pattern, the 4-lobed one from notebook 05. The model gives it two faint overtones at the ring ratios from notebook 05, with 12% and 4% of the fundamental's amplitude. The mallet goes round for 8 s, building the tone up over the first 2 s. When it stops, the bowl rings down slowly.

In [ ]:
one_bowl = crystal_bowl(BOWL_1_HZ, T)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 3.8),
                                    gridspec_kw={"width_ratios": [1.4, 1, 1]})
ax1.plot(T, one_bowl, color=BLUE, linewidth=0.2)
ax1.plot(T, bowl_envelope(T) * sum(OVERTONE_LEVELS), color=ORANGE, linewidth=1.5,
         label="envelope")
ax1.set_xlabel("time (s)")
ax1.set_ylabel("sound pressure")
ax1.set_title("12 s of one bowl: swell, hold, ring down", fontsize=10)
ax1.legend(fontsize=8, loc="upper right")

window = (T > 3) & (T < 3 + 4 / BOWL_1_HZ)
ax2.plot((T[window] - 3) * 1000, one_bowl[window], color=BLUE)
ax2.set_xlabel("time (ms)")
ax2.set_title("Four cycles, close up", fontsize=10)

steady = (T > 3) & (T < 7)
spectrum = np.abs(np.fft.rfft(one_bowl[steady] * np.hanning(steady.sum()))) ** 2
freqs = np.fft.rfftfreq(steady.sum(), 1 / AUDIO_RATE)
ax3.plot(freqs, 10 * np.log10(spectrum / spectrum.max() + 1e-12), color=BLUE,
         linewidth=1)
for ratio in OVERTONE_RATIOS:
    ax3.text(BOWL_1_HZ * ratio, 3, f"{BOWL_1_HZ * ratio:.0f}", ha="center", fontsize=8,
             color=ORANGE)
ax3.set_xlim(0, 1700)
ax3.set_ylim(-80, 8)
ax3.set_xlabel("frequency (Hz)")
ax3.set_ylabel("dB")
ax3.set_title("Spectrum while held", fontsize=10)
plt.tight_layout()
plt.show()
play(one_bowl)

**What the plots show**

- **Left:** 12 s of one bowl. The tone swells over 2 s as the mallet builds it up, holds steady until 8 s, then rings down. By 12 s it has fallen to a tenth of its strength (−20 dB).
- **Middle:** four cycles, 15 ms in all. The wave is almost a pure sine. The small bumps come from the faint overtones.
- **Right:** the spectrum while the tone is held. One strong line at 262 Hz, then 740 Hz at −18 dB and 1418 Hz at −28 dB. A crystal bowl in this model is close to a single pure tone.

### Watching the bowl vibrate

The animation shows the bowl's 4-lobed pattern, the one that carries almost all of its sound. Two things are changed so it can be seen:

- **The motion is enlarged many thousands of times.** A real rim moves by far less than a millimetre.
- **Time is slowed about 260 times.** One cycle takes 1 s here, against 3.8 ms in the real bowl.

The bending grows from nothing at the base to its largest at the rim. Four points round the rim, at 45°, 135°, 225° and 315°, never move. These are the **nodes**. Between them the rim swings in and out, with opposite lobes in step.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

ENLARGE = 0.15          # drawing units, rim displacement at the peak of a cycle
FRAMES_PER_CYCLE = 24


def bowl_shape(n, bend):
    # A hemispherical shell open at the top: psi = 0 at the base, pi/2 at the rim.
    # Pattern n bends the wall in and out, growing toward the rim.
    psi, phi = np.meshgrid(np.linspace(0, np.pi / 2, 16), np.linspace(0, 2 * np.pi, 73))
    w = bend * np.sin(psi) ** 2 * np.cos(n * phi)
    radius = np.sin(psi) * (1 + w)
    return radius * np.cos(phi), radius * np.sin(phi), -np.cos(psi), w


def draw_rim(ax, n, bend, colour):
    phi = np.linspace(0, 2 * np.pi, 361)
    radius = 1 + bend * np.cos(n * phi)
    ax.plot(radius * np.cos(phi), radius * np.sin(phi), color=colour, linewidth=2.5)


fig = plt.figure(figsize=(10, 4.6))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax_top = fig.add_subplot(1, 2, 2)
nodes = (np.arange(4) + 0.5) * np.pi / 2


def draw_bowl_frame(frame):
    bend = ENLARGE * np.cos(2 * np.pi * frame / FRAMES_PER_CYCLE)
    x, y, z, w = bowl_shape(2, bend)
    ax3d.clear()
    ax3d.plot_surface(x, y, z, facecolors=plt.cm.RdBu_r(0.5 + w / (2 * ENLARGE)),
                      linewidth=0, antialiased=False, shade=False)
    ax3d.set_box_aspect((1, 1, 0.55))
    ax3d.set_xlim(-1.15, 1.15)
    ax3d.set_ylim(-1.15, 1.15)
    ax3d.set_zlim(-1.0, 0.1)
    ax3d.view_init(elev=28, azim=-60)
    ax3d.set_axis_off()
    ax3d.set_title("The bowl (red: wall pushed out, blue: pulled in)", fontsize=10)

    ax_top.clear()
    circle = np.linspace(0, 2 * np.pi, 200)
    ax_top.plot(np.cos(circle), np.sin(circle), color=GREY, linestyle=":", linewidth=1)
    draw_rim(ax_top, 2, bend, BLUE)
    ax_top.scatter(np.cos(nodes), np.sin(nodes), s=40, color="k", zorder=4, label="nodes")
    ax_top.set_aspect("equal")
    ax_top.set_xlim(-1.3, 1.3)
    ax_top.set_ylim(-1.3, 1.3)
    ax_top.axis("off")
    ax_top.legend(loc="lower right", fontsize=8)
    ax_top.set_title(f"The rim from above, {frame / FRAMES_PER_CYCLE:.2f} of a cycle",
                     fontsize=10)


BOWL_ANIMATION = FuncAnimation(fig, draw_bowl_frame, frames=FRAMES_PER_CYCLE, interval=1000 / FRAMES_PER_CYCLE)
plt.close(fig)
display(HTML(BOWL_ANIMATION.to_jshtml(default_mode="loop")))

**What the animation shows**

- **Left:** the bowl in 3D. Red bands are where the wall is pushed out and blue bands where it is pulled in. The colours swap every half cycle. The bands are strongest at the rim and fade to nothing at the base, which stays still.
- **Right:** the rim from above. At the start of a cycle it is stretched side to side. Half a cycle later it is stretched front to back. The four black nodes stay on the dotted round rim the whole time.

This in-and-out motion pushes on the air and makes the sound. Opposite lobes push out together while the lobes between them pull in, so the air around the bowl is pushed and pulled in a four-part pattern.

## 2. Two bowls: beats

Now add a second bowl, a little sharp. Two tones at $f_1$ and $f_2$ add up to

$$\cos(2\pi f_1 t) + \cos(2\pi f_2 t) = 2\cos\!\left(2\pi \frac{f_2 - f_1}{2} t\right)\cos\!\left(2\pi \frac{f_1 + f_2}{2} t\right)$$

The second factor is a tone at the average pitch. The first factor slowly changes its strength. The two bowls drift in and out of step: in step they add and the sound is loud, half a cycle apart they cancel and it is quiet. Loudness peaks $f_2 - f_1$ times a second. This is the **beat rate**.

This is the same constructive and destructive interference as in notebook 01. There it happened across space, here it happens over time.

The sliders set how far the second bowl is from the first and how strong it is.

In [ ]:
@interact(
    detune_hz=FloatSlider(value=1.4, min=0.0, max=30.0, step=0.1,
                          description="gap (Hz)", continuous_update=False,
                          style=SLIDER_STYLE),
    strength=FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                         description="bowl 2 level", continuous_update=False,
                         style=SLIDER_STYLE),
)
def show_beats(detune_hz, strength):
    bowl_1 = crystal_bowl(BOWL_1_HZ, T)
    bowl_2 = strength * crystal_bowl(BOWL_1_HZ + detune_hz, T)
    both = bowl_1 + bowl_2
    t_env = np.linspace(0, 12, 4000)
    envelope = (np.abs(1 + strength * np.exp(2j * np.pi * detune_hz * t_env))
                * bowl_envelope(t_env))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6.4),
                                   gridspec_kw={"height_ratios": [1.3, 1]})
    ax1.plot(T, both, color=BLUE, linewidth=0.2)
    ax1.plot(t_env, envelope, color=ORANGE, linewidth=1.5, label="loudness envelope")
    ax1.set_xlim(0, 12)
    ax1.set_xlabel("time (s)")
    ax1.set_ylabel("sound pressure")
    rate = f"{detune_hz:.1f} beats per second" if detune_hz > 0 else "no beats"
    ax1.set_title(f"Bowls at {BOWL_1_HZ:.1f} Hz and {BOWL_1_HZ + detune_hz:.1f} Hz: {rate}",
                  fontsize=10)
    ax1.legend(fontsize=8, loc="upper right")

    window = (T > 4) & (T < 6)
    ax2.plot(T[window], bowl_1[window], color=GREY, linewidth=0.6, label="bowl 1")
    ax2.plot(T[window], bowl_2[window], color=AQUA, linewidth=0.6, label="bowl 2")
    ax2.plot(T[window], both[window], color=BLUE, linewidth=0.8, label="both")
    ax2.set_xlim(4, 6)
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("sound pressure")
    ax2.set_title("2 s close up: each bowl on its own, and the sum", fontsize=10)
    ax2.legend(fontsize=8, loc="upper right", ncol=3)
    plt.tight_layout()
    plt.show()
    play(both)

**What the plots show** (defaults: bowls at 261.6 Hz and 263.0 Hz, equal strength)

- **Top:** 12 s of both bowls. The sound pulses 1.4 times a second, rising to twice one bowl's strength and falling almost to silence between pulses. The orange envelope is the formula above.
- **Bottom:** 2 s close up. Grey and green are each bowl alone, at steady strength. Blue is the sum. Near 4.3 s and 5.0 s the bowls are in step and the sum is twice as large. Near 4.65 s they are half a cycle apart and the sum nearly vanishes.

**Try this:**

- **Gap 6 Hz:** six pulses a second, a fast tremolo.
- **Gap 25 Hz:** the envelope is too fast to follow as separate pulses. Most listeners hear this as a rough, buzzing tone rather than a throb.
- **Gap 0:** the bowls stay in step for ever. One steady, louder tone.
- **Bowl 2 level 0.3:** the envelope only swings between 0.7 and 1.3. A weak second bowl gives a shallow throb that never falls silent.

### Watching two bowls beat

Two rims from above, each in its 4-lobed pattern, with the sound they make below. Here the gap in pitch is enlarged as well: the second bowl vibrates 9 times for every 8 of the first, so they drift through one beat in 8 cycles. Real bowls 1.4 Hz apart take about 187 cycles to do the same.

Watch the two rims. At the start they swing in step and the summed wave is large. Halfway through, one is pushed out while the other is pulled in, and the sum nearly vanishes.

In [ ]:
CYCLES = 8                                  # cycles of bowl 1 per beat
FRAMES = CYCLES * 12
beat_t = np.linspace(0, CYCLES, 1600)       # time in cycles of bowl 1
wave_1 = np.cos(2 * np.pi * beat_t)
wave_2 = np.cos(2 * np.pi * beat_t * (CYCLES + 1) / CYCLES)

fig = plt.figure(figsize=(10, 5.6))
grid = fig.add_gridspec(2, 2, height_ratios=[1.2, 1])
ax_1, ax_2 = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])
ax_wave = fig.add_subplot(grid[1, :])


def draw_beat_frame(frame):
    t = CYCLES * frame / FRAMES
    phases = [2 * np.pi * t, 2 * np.pi * t * (CYCLES + 1) / CYCLES]
    for ax, phase, colour, name in [(ax_1, phases[0], GREY, "bowl 1"),
                                    (ax_2, phases[1], AQUA, "bowl 2 (9/8 the pitch)")]:
        ax.clear()
        circle = np.linspace(0, 2 * np.pi, 200)
        ax.plot(np.cos(circle), np.sin(circle), color=GREY, linestyle=":", linewidth=1)
        draw_rim(ax, 2, ENLARGE * np.cos(phase), colour)
        ax.set_aspect("equal")
        ax.set_xlim(-1.3, 1.3)
        ax.set_ylim(-1.3, 1.3)
        ax.axis("off")
        ax.set_title(name, fontsize=10)

    ax_wave.clear()
    ax_wave.plot(beat_t, wave_1, color=GREY, linewidth=0.8, label="bowl 1")
    ax_wave.plot(beat_t, wave_2, color=AQUA, linewidth=0.8, label="bowl 2")
    ax_wave.plot(beat_t, wave_1 + wave_2, color=BLUE, linewidth=1.5, label="both")
    ax_wave.plot(beat_t, np.abs(2 * np.cos(np.pi * beat_t / CYCLES)), color=ORANGE,
                 linewidth=1, linestyle="--", label="envelope")
    ax_wave.axvline(t, color="k", linewidth=1.5)
    ax_wave.set_xlim(0, CYCLES)
    ax_wave.set_ylim(-2.3, 2.3)
    ax_wave.set_xlabel("time (cycles of bowl 1)")
    ax_wave.set_ylabel("sound pressure")
    ax_wave.legend(fontsize=8, loc="upper center", ncol=4)


BEAT_ANIMATION = FuncAnimation(fig, draw_beat_frame, frames=FRAMES, interval=80)
plt.close(fig)
display(HTML(BEAT_ANIMATION.to_jshtml(default_mode="loop")))

**What the animation shows**

- **Top:** the two rims. At the start both are stretched side to side at the same moment. Bowl 2 gains a little on bowl 1 every cycle. At 4 cycles, halfway, bowl 1 is stretched side to side while bowl 2 is stretched front to back: they are exactly out of step. By 8 cycles they are back in step.
- **Bottom:** grey and green are each bowl's sound, blue is their sum, and the black line marks the current moment. The sum is twice as large at the start and end and almost zero in the middle. The orange dashed envelope traces one full beat.

## 3. Two bowls around a listener

In a room each ear hears both bowls. Sound from the left bowl reaches the left ear slightly sooner than the right ear, and the reverse for the right bowl. These tiny delays shift the beat at one ear relative to the other. The model puts the ears 0.18 m apart, with the bowls 2 m to either side and 1 m in front. It ignores the head's shadow and the room's reflections. The slider moves the listener sideways.

The audio is in stereo. The last player is different: it sends bowl 1 only to the left ear and bowl 2 only to the right ear. No sound wave beats there, because each ear hears one steady tone. Many people still sense a faint pulse, made by the brain comparing the two ears. This is a **binaural beat**, and it needs headphones. In a room both ears hear both bowls, so what you hear there are ordinary beats.

In [ ]:
EAR_SPACING = 0.18                       # m
BOWLS = [(BOWL_1_HZ, np.array([-2.0, 1.0])), (BOWL_2_HZ, np.array([2.0, 1.0]))]


def ear_positions(listener_x):
    return {"left": np.array([listener_x - EAR_SPACING / 2, 0.0]),
            "right": np.array([listener_x + EAR_SPACING / 2, 0.0])}


def at_ear(ear):
    # Both bowls as heard at one ear: delayed by r / c and weakened as 1 / r
    sound = np.zeros_like(T)
    for f, position in BOWLS:
        r = np.linalg.norm(position - ear)
        sound += crystal_bowl(f, T, delay_s=r / SPEED_OF_SOUND) / r
    return sound


def beat_envelope(ear, t):
    # Strength of the two fundamentals added together at one ear
    (f1, p1), (f2, p2) = BOWLS
    r1, r2 = np.linalg.norm(p1 - ear), np.linalg.norm(p2 - ear)
    phase = 2 * np.pi * ((f2 - f1) * t - f2 * r2 / SPEED_OF_SOUND + f1 * r1 / SPEED_OF_SOUND)
    return np.abs(1 / r1 + np.exp(1j * phase) / r2)


@interact(
    listener_x=FloatSlider(value=0.0, min=-1.5, max=1.5, step=0.05,
                           description="listener x (m)", continuous_update=False,
                           style=SLIDER_STYLE),
)
def show_two_ears(listener_x):
    ears = ear_positions(listener_x)
    t_env = np.linspace(4, 7, 3000)
    envelopes = {name: beat_envelope(ear, t_env) for name, ear in ears.items()}
    beat_period = 1 / (BOWL_2_HZ - BOWL_1_HZ)
    lag = (t_env[np.argmax(envelopes["right"][:1500])]
           - t_env[np.argmax(envelopes["left"][:1500])])
    lag = (lag + beat_period / 2) % beat_period - beat_period / 2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.4),
                                   gridspec_kw={"width_ratios": [1, 1.7]})
    for (f, position), colour in zip(BOWLS, [GREY, AQUA]):
        ax1.scatter(*position, s=300, color=colour, edgecolor="k", zorder=3)
        ax1.annotate(f"{f:.1f} Hz", position, xytext=(0, 14), textcoords="offset points",
                     ha="center")
    for (name, ear), colour in zip(ears.items(), [BLUE, ORANGE]):
        ax1.scatter(*ear, s=60, color=colour, zorder=4, label=f"{name} ear")
    ax1.add_patch(plt.Circle((listener_x, 0), EAR_SPACING / 2, fill=False, color="k"))
    ax1.set_aspect("equal")
    ax1.set_xlim(-2.8, 2.8)
    ax1.set_ylim(-0.8, 1.8)
    ax1.set_xlabel("x (m)")
    ax1.set_ylabel("y (m)")
    ax1.legend(fontsize=8, loc="lower right")
    ax1.set_title("Two bowls and a listener, from above", fontsize=10)

    for (name, env), colour in zip(envelopes.items(), [BLUE, ORANGE]):
        ax2.plot(t_env, env / env.max(), color=colour, label=f"{name} ear")
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("loudness envelope (peak = 1)")
    when = "before" if lag < 0 else "after"
    ax2.set_title(f"Beat at each ear: the right ear's loud moments come {abs(lag) * 1000:.0f} ms "
                  f"{when} the left's ({abs(lag) / beat_period:.2f} of a beat)", fontsize=10)
    ax2.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.show()

    print("In the room (stereo, both ears hear both bowls):")
    play(np.vstack([at_ear(ears["left"]), at_ear(ears["right"])]))
    print("Binaural (headphones, each ear hears one bowl):")
    play(np.vstack([crystal_bowl(BOWL_1_HZ, T), crystal_bowl(BOWL_2_HZ, T)]))

**What the plots show** (default: listener midway between the bowls)

- **Left:** the layout from above. The bowls are 2 m to each side and 1 m in front, the ears 0.18 m apart.
- **Right:** the throb at each ear. Both ears beat 1.4 times a second and dip almost to silence. The right ear's loud moments come 176 ms before the left's, a quarter of a beat. The offset comes from the half-millisecond differences in travel time to each ear. It can make the throb seem to move across the head.

**Try this:**

- **Listener x = 0.5 m**, closer to the right bowl. That bowl is now louder at both ears, so the dips no longer reach silence: the depth falls to about 80%. The offset between the ears hardly changes.
- **Listener x = −1 m**, close to the left bowl. The depth falls to about 60%. Near one bowl, that bowl dominates and the throb fades.
- **Compare the two players.** The first is the room, with both bowls reaching both ears. The second is binaural: each ear hears one steady tone. Any pulse heard in the second one is made inside the head.

## 4. Two bowls in the dome

Notebook 02 found that a steady note at the dome's centre is boosted or cut depending on where it falls on a comb of resonances 20.5 Hz apart. The peaks are narrow: at α = 0.1 a peak is about 0.34 Hz wide. Two bowls 1.4 Hz apart can therefore land at very different places on the comb, and the dome changes how strong each one is.

Beats are deepest when the two notes are equally strong. If the dome boosts one bowl and cuts the other, the louder bowl dominates and the throb becomes shallow. The **beat depth** below measures this: 100% means the sound falls to silence between peaks, 0% means a steady tone.

As in notebook 02, both bowls and the listener are treated as being at the centre. The audio uses the same echo rule as notebook 02's "Hear it" section.

In [ ]:
def centre_gain(f, alpha):
    # Steady-state amplitude gain at the centre (notebook 02)
    return np.abs(1 / (1 + np.sqrt(1 - alpha) * np.exp(2j * np.pi * f * ROUND_TRIP_S)))


def centre_echoes(signal, alpha):
    # y[n] = x[n] - r y[n - D]: every 49 ms a flipped, weakened copy comes back
    feedback = np.zeros(round(ROUND_TRIP_S * AUDIO_RATE) + 1)
    feedback[0], feedback[-1] = 1.0, np.sqrt(1 - alpha)
    return lfilter([1.0], feedback, signal)


def beat_depth(a1, a2):
    return 2 * min(a1, a2) / (a1 + a2)


@interact(
    bowl_1_hz=FloatSlider(value=BOWL_1_HZ, min=250.0, max=280.0, step=0.1,
                          description="bowl 1 (Hz)", continuous_update=False,
                          style=SLIDER_STYLE),
    detune_hz=FloatSlider(value=1.4, min=0.2, max=5.0, step=0.1, description="gap (Hz)",
                          continuous_update=False, style=SLIDER_STYLE),
    alpha=FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description="wall α",
                      continuous_update=False, style=SLIDER_STYLE),
)
def show_bowls_in_dome(bowl_1_hz, detune_hz, alpha):
    f1, f2 = bowl_1_hz, bowl_1_hz + detune_hz
    g1, g2 = centre_gain(f1, alpha), centre_gain(f2, alpha)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.4))
    f = np.linspace(245, 290, 6000)
    ax1.plot(f, 20 * np.log10(centre_gain(f, alpha)), color=GREY, linewidth=1)
    for freq, gain, colour, name in [(f1, g1, BLUE, "bowl 1"), (f2, g2, ORANGE, "bowl 2")]:
        ax1.axvline(freq, color=colour, linewidth=1.2)
        ax1.scatter([freq], [20 * np.log10(gain)], s=60, color=colour, zorder=4,
                    label=f"{name}: {freq:.1f} Hz, {20 * np.log10(gain):+.1f} dB")
    ax1.set_xlabel("frequency (Hz)")
    ax1.set_ylabel("gain at the centre (dB)")
    ax1.set_title(f"Where the two bowls land on the dome's comb (α = {alpha:.2f})",
                  fontsize=10)
    ax1.legend(fontsize=8, loc="upper right")

    t = np.linspace(0, 3, 3000)
    dry = np.abs(1 + np.exp(2j * np.pi * detune_hz * t))
    dome = np.abs(g1 + g2 * np.exp(2j * np.pi * detune_hz * t))
    ax2.plot(t, dry / dry.max(), color=GREY, linestyle="--",
             label=f"without the dome: depth {beat_depth(1, 1):.0%}")
    ax2.plot(t, dome / dome.max(), color=BLUE,
             label=f"at the centre: depth {beat_depth(g1, g2):.0%}")
    ax2.set_ylim(0, 1.15)
    ax2.set_xlabel("time (s), steady state")
    ax2.set_ylabel("loudness envelope (peak = 1)")
    ax2.set_title("The throb with and without the dome", fontsize=10)
    ax2.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.show()

    both = crystal_bowl(f1, T) + crystal_bowl(f2, T)
    print("Without the dome:")
    play(both)
    print("At the centre of the dome:")
    play(centre_echoes(both, alpha))

**What the plots show** (defaults: bowls at 261.6 Hz and 263.0 Hz, α = 0.1)

- **Left:** the dome's comb from notebook 02 between 245 and 290 Hz, with peaks at 255.8 Hz and 276.2 Hz. Both bowls fall in the dip between them and are cut by similar amounts, −3.7 dB and −4.8 dB.
- **Right:** the steady throb with and without the dome. Because the dome cuts both bowls about equally, the throb barely changes: 93% deep instead of 100%.

**Try this:**

- **Bowl 1 at 255.8 Hz**, right on a comb peak. Bowl 1 gets +25.5 dB and bowl 2, 1.4 Hz higher, gets +7.3 dB. Bowl 1 now dominates and the throb nearly disappears, to 22% depth. Moving one bowl by a few hertz decides whether the dome keeps the beat or erases it.
- **Bowl 1 at 255.1 Hz.** The two bowls straddle the peak at +13.9 and +12.8 dB. Both are boosted and the throb stays deep (94%).
- **Then set the gap to 0.7 Hz.** Bowl 2 now lands on the peak and the depth falls to 41%.
- **α = 0.4 with bowl 1 at 255.8 Hz.** A more absorbent wall gives broader, lower peaks, and the depth is back up to 67%.

In the audio, the dome's effect builds up over the first few seconds of rubbing, because the resonances take time to ring up.

## Summary

| Idea | Formula | Example |
|---|---|---|
| Beat rate | $f_2 - f_1$ | 1.4 pulses a second for 261.6 and 263.0 Hz |
| Beat depth | $2\min(a_1, a_2)/(a_1 + a_2)$ | 100% for equal bowls, 46% if one is at 0.3 strength |
| Offset between the ears | difference in travel times | 176 ms, a quarter of a beat, for bowls 2 m to each side |
| Binaural beat | one tone per ear | needs headphones, does not occur in a room |
| The dome | each bowl gets its own comb gain | 93% depth with both in a dip, 22% with one on a peak (α = 0.1) |

1. A rubbed crystal bowl in this model is close to a pure tone, with faint overtones.
2. Two bowls a little apart in pitch drift in and out of step, and the sound throbs at the difference in their frequencies. Below a few hertz this is a slow pulse. Above about 20 Hz it turns into roughness.
3. The throb is deepest when both bowls are equally loud. Standing near one bowl makes it shallower.
4. In a room each ear hears both bowls, slightly offset in time, so the throb reaches the two ears at different moments. Binaural beats are a different thing that needs headphones.
5. At the dome's centre each bowl is boosted or cut by its own amount. A bowl that lands on a resonance peak can drown out its partner and erase the throb.

**Back to:** [00 Overview](00_overview.md)